# 📗 Cypher 기초: CREATE·MATCH·RETURN·SET·DELETE

지난 시간에는 Neo4j 를 설치하고 완성된 그래프를 **눈으로 탐색**했습니다. 이번 시간에는 그래프에게 말을 거는 언어 **Cypher** 를 직접 씁니다. Cypher 는 **그림을 그리듯** 쓰는 쿼리 언어입니다. 동그라미(노드)를 `()` 로, 화살표(관계)를 `-[:관계]->` 로 그리면 그게 곧 쿼리가 됩니다.

오늘 도메인은 스타트업 **노바랩스**의 조직도입니다. 직원이 어느 팀에 속하고 어느 프로젝트에 배정됐는지를 그래프로 만들고(**CREATE**), 다시 찾고(**MATCH·RETURN**), 바뀐 값을 고치고(**SET·REMOVE**), 잘못 만든 것은 지웁니다(**DELETE**).

## ⏪ 복습: 지난 시간까지

- **그래프DB(LPG)**: 데이터를 **노드**(개체)·**관계**(연결)·**속성**(값)으로 담는 모델입니다.
- **노드에는 레이블**(`Employee` 같은 종류 이름)과 **속성**(`name`, `role`)이 붙습니다.
- **관계에는 방향과 종류**가 있습니다(`WORKS_IN` 은 직원에서 팀으로 향함).

**오늘의 목표**

**1. 생성**
- [ ] (1-1) **CREATE** 로 노드를 만들고, `RETURN` 을 이어 붙여 방금 만든 것을 바로 돌려받는다.
- [ ] (1-2) 속성의 **자료형**(정수·실수·문자열·불리언·리스트·날짜)을 구분해 적고, 날짜는 `date()` 로 적어야 연·월을 꺼내고 기간을 잴 수 있다는 것을 안다.
- [ ] (1-3) 한 노드에 **레이블을 여러 개** 붙이고, 레이블을 나란히 적은 패턴이 **AND** 라는 것을 안다.
- [ ] (1-4) 두 노드를 찾아 **관계**로 잇는다.
- [ ] (1-5) **관계에도 속성**을 붙이고(`-[r:종류 {속성: 값}]->`), 그 값을 꺼낸다.
- [ ] (1-6) 노드와 관계를 **한 문장**으로 만들고, 그 형태를 언제 쓰면 안 되는지 안다.

**2. 조회**
- [ ] (2-1) **MATCH … RETURN … AS 별칭** 으로 찾아서 값을 돌려받는다.
- [ ] (2-2) 속성을 골라 받을 때와 노드를 통째로 받을 때의 **모양 차이**를 안다.

**3. 수정**
- [ ] (3-1) **SET** 으로 있던 값을 덮어쓰고, 없던 속성을 새로 더한다.
- [ ] (3-2) **REMOVE** 로 속성 한 칸이나 레이블만 떼어 낸다.

**4. 삭제**
- [ ] (4-1) **DELETE** 로 노드 또는 관계를 골라 지운다.
- [ ] (4-2) 관계가 붙은 노드는 **DETACH DELETE** 로 지운다.

아래 준비 셀 4개를 위에서부터 실행하세요. Cypher 는 `run_cypher("...")` 헬퍼로 실행합니다. 결과는 **dict 의 리스트**로 돌아옵니다(그 뒤 파이썬으로 다룹니다).

**연결이 안 되면**: `.env` 의 `NEO4J_URI`·`NEO4J_USER`·`NEO4J_PASSWORD` 가 지금 켜져 있는 실습용 데이터베이스의 값인지 먼저 확인하세요. `ServiceUnavailable` 은 **DB 가 꺼져 있거나 포트가 다른 것**, `AuthError` 는 **아이디·비밀번호가 다른 것**입니다.

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. 반드시 "실습 전용" DB 여야 합니다(아래 실습이 그래프를 지웁니다).
# 앞으로 모든 Cypher 는 run_cypher("쿼리", 파라미터=값) 으로 실행하고, 결과는 dict 리스트로 옵니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # 지우기 규칙 위반 에러

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러 없이 기본값으로 넘어간다. 마지막 줄에 찍히는 주소를 눈으로 꼭 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 여기까지 찍히면 준비 완료

> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 지난 단원에서 브라우저로 적재한 **Movies 예제 그래프와 그때 푼 과제 결과도 함께 사라집니다.** 되돌릴 수 없으니, `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요. Movies 를 남기고 싶다면 실습용 인스턴스를 따로 하나 만들어 그 접속 정보를 `.env` 에 넣으면 됩니다(지웠더라도 day28 폴더의 `data/movies_setup.cypher` 로 다시 적재할 수 있습니다).

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙어 있는 관계까지 함께 지우라는 뜻입니다(교안_01 마지막 절에 나옵니다).
run_cypher("MATCH (n) DETACH DELETE n")
# 확인: MATCH (n) RETURN n 은 남은 노드를 한 줄씩 돌려주므로 그 행 수가 곧 노드 개수다
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

오늘 만들고 조회할 그래프의 전체 모습입니다. 직원 5명, 팀 2개, 프로젝트 2개가 세 종류의 관계로 이어져 있습니다.

<img src="images/org-graph-overview.png" width="820">

> 아래 셀은 오늘 조회 실습에 쓸 **완성된 조직 그래프**를 미리 적재합니다. 적재에 쓰인 `CREATE` 는 바로 이 시간에 배웁니다. 지금은 실행만 하고, 우리가 배운 문법으로 이 그래프를 조회·확장해 봅니다.

In [ ]:
# [제공 코드] 스타트업 "노바랩스" 조직 그래프 적재: 이 셀은 실행만 하세요.
# 직원·팀·프로젝트 노드와 소속(WORKS_IN)·배정(ASSIGNED_TO)·소유(OWNS) 관계를 CREATE 로 만듭니다.
# 1) 넣을 값을 파이썬 리스트로 먼저 적어 둔다. years 는 뒤에서 조건 비교에 쓰인다
employees = [
    {"name": "김서준", "role": "백엔드", "years": 5, "team": "개발팀", "project": "결제시스템"},
    {"name": "정민재", "role": "백엔드", "years": 2, "team": "개발팀", "project": "결제시스템"},
    {"name": "박도윤", "role": "데이터", "years": 4, "team": "개발팀", "project": "결제시스템"},
    {"name": "이하은", "role": "프론트엔드", "years": 3, "team": "개발팀", "project": "앱개편"},
    {"name": "최지우", "role": "디자인", "years": 1, "team": "디자인팀", "project": "앱개편"},
]
projects = [
    {"name": "결제시스템", "status": "진행", "deadline": "2026-12-31"},
    {"name": "앱개편", "status": "완료", "deadline": "2026-03-31"},
]

team_projects = [("개발팀", "결제시스템"), ("디자인팀", "앱개편")]  # 팀이 맡은(소유한) 프로젝트

# 2) 프로젝트 노드부터 만든다. 마감일은 글자가 아니라 date() 로 만든 날짜 값으로 담는다
for p in projects:
    run_cypher(
        "CREATE (:Project {name: $name, status: $status, deadline: date($deadline)})",
        name=p["name"], status=p["status"], deadline=p["deadline"],
    )
# 3) 팀 노드. 팀은 이름뿐이라 속성이 하나다
for t in ["개발팀", "디자인팀"]:
    run_cypher("CREATE (:Team {name: $name})", name=t)
# 4) 팀 -> 프로젝트 관계(OWNS): 오늘 체인 패턴에서 두 번째 화살표로 쓰인다
for team, proj in team_projects:
    run_cypher(
        "MATCH (t:Team {name: $team}), (p:Project {name: $proj}) CREATE (t)-[:OWNS]->(p)",
        team=team, proj=proj,
    )
# 5) 직원 한 명마다 노드 1개 + 관계 2개를 만든다. 관계는 양쪽 노드가 이미 있어야 그을 수 있어 순서가 중요하다
for e in employees:
    run_cypher(
        "CREATE (:Employee {name: $name, role: $role, years: $years})",
        name=e["name"], role=e["role"], years=e["years"],
    )
    # 소속: 직원 -> 팀
    run_cypher(
        "MATCH (e:Employee {name: $name}), (t:Team {name: $team}) "
        "CREATE (e)-[:WORKS_IN]->(t)",
        name=e["name"], team=e["team"],
    )
    # 배정: 직원 -> 프로젝트
    run_cypher(
        "MATCH (e:Employee {name: $name}), (p:Project {name: $project}) "
        "CREATE (e)-[:ASSIGNED_TO]->(p)",
        name=e["name"], project=e["project"],
    )

print("적재한 노드 수:", len(run_cypher("MATCH (n) RETURN n")))   # 직원 5 + 팀 2 + 프로젝트 2

시연은 위의 **노바랩스 조직** 그래프로 하고, `🖐️ 함께 따라하기` 는 아래에서 적재하는 **캠퍼스라운지 동아리** 그래프로 합니다. 배운 문법을 **다른 데이터에 옮겨 써 보는 것**이 따라하기의 목적이라 데이터를 나눠 두었습니다. 두 그래프는 레이블이 달라 한 데이터베이스에 함께 있어도 서로 섞이지 않습니다.

| 노드 | 속성 | 관계 |
|---|---|---|
| `Student`(학생) | `name`, `major`(전공), `grade`(학년) | `BELONGS_TO`: 학생에서 동아리로 |
| `Club`(동아리) | `name` | `HOSTS`: 동아리에서 행사로 |
| `Event`(행사) | `name`, `status` | `JOINS`: 학생에서 행사로 |

아래 셀이 적재할 동아리 그래프의 전체 모습입니다. 학생 5명, 동아리 2개, 행사 3개가 세 종류의 관계로 이어져 있습니다. 조직 그래프와 **레이블만 다르고 짜임은 같습니다.**

<img src="images/club-graph-overview.png" width="820">

In [ ]:
# [제공 코드] 따라하기용 "캠퍼스라운지" 동아리 그래프 적재: 이 셀은 실행만 하세요.
# 학생·동아리·행사 노드와 소속(BELONGS_TO)·주최(HOSTS)·참가(JOINS) 관계를 만듭니다.
# 위 조직 그래프와 만드는 순서가 같고, 레이블이 달라 한 DB 에 함께 있어도 섞이지 않습니다.
# 1) 넣을 값을 파이썬 리스트로 먼저 적어 둔다. grade(학년)는 뒤에서 조건 비교에 쓰인다
students = [
    {"name": "윤도현", "major": "경영",   "grade": 3, "club": "사진동아리", "event": "봄전시회"},
    {"name": "서지안", "major": "컴퓨터", "grade": 1, "club": "사진동아리", "event": "봄전시회"},
    {"name": "노태윤", "major": "디자인", "grade": 4, "club": "사진동아리", "event": "출사모임"},
    {"name": "임하늘", "major": "경영",   "grade": 2, "club": "밴드동아리", "event": "가을공연"},
    {"name": "구본재", "major": "컴퓨터", "grade": 3, "club": "밴드동아리", "event": "가을공연"},
]
events = [
    {"name": "봄전시회", "status": "모집중"},
    {"name": "출사모임", "status": "모집중"},
    {"name": "가을공연", "status": "마감"},
]

club_events = [("사진동아리", "봄전시회"), ("사진동아리", "출사모임"), ("밴드동아리", "가을공연")]

# 2) 행사 노드
for e in events:
    run_cypher("CREATE (:Event {name: $name, status: $status})", name=e["name"], status=e["status"])
# 3) 동아리 노드
for c in ["사진동아리", "밴드동아리"]:
    run_cypher("CREATE (:Club {name: $name})", name=c)
# 4) 동아리 -> 행사 관계(HOSTS): 사진동아리는 행사를 둘 여니 화살표가 두 개 나간다
for club, event in club_events:
    run_cypher(
        "MATCH (c:Club {name: $club}), (v:Event {name: $event}) CREATE (c)-[:HOSTS]->(v)",
        club=club, event=event,
    )
# 5) 학생 한 명마다 노드 1개 + 관계 2개(소속·참가)
for s in students:
    run_cypher(
        "CREATE (:Student {name: $name, major: $major, grade: $grade})",
        name=s["name"], major=s["major"], grade=s["grade"],
    )
    # 소속: 학생 -> 동아리
    run_cypher(
        "MATCH (s:Student {name: $name}), (c:Club {name: $club}) "
        "CREATE (s)-[:BELONGS_TO]->(c)",
        name=s["name"], club=s["club"],
    )
    # 참가: 학생 -> 행사
    run_cypher(
        "MATCH (s:Student {name: $name}), (v:Event {name: $event}) "
        "CREATE (s)-[:JOINS]->(v)",
        name=s["name"], event=s["event"],
    )

print("적재한 학생 수:", len(run_cypher("MATCH (s:Student) RETURN s")))   # Student 레이블만 세므로 조직 그래프는 안 잡힌다

---
# 1. 생성: 그래프에 데이터 넣기

그래프에 무엇을 넣는 일은 이렇게 이어집니다. **노드 하나**를 만들고(1-1), 그 안에 담기는 값의 **자료형**을 가리고(1-2), 한 노드에 **레이블을 여러 개** 붙일 수 있다는 것을 보고(1-3), 두 노드를 **관계**로 잇고(1-4), 그 관계에 **값을 담고**(1-5), 마지막으로 **한 문장에 몰아서** 만듭니다(1-6). 만드는 낱말은 내내 `CREATE` 하나뿐입니다.

## 1-1. 노드 만들기

### 왜 필요할까요?
그래프에 데이터를 넣는 첫걸음은 **노드 하나를 만드는 것**입니다. 새 직원이 들어오면 그 사람을 나타내는 노드를 하나 만들어야 하죠.

### 문법: 노드 패턴
노드는 소괄호로 그립니다. 안에 **변수·레이블·속성**을 담습니다.

```text
CREATE (:Employee {name: '한소희', role: '백엔드'})
```

- `(...)` : 노드 하나(동그라미)
- `:Employee` : **레이블**(노드의 종류). 대문자로 시작하는 게 관례입니다.
- `{name: '한소희', role: '백엔드'}` : **속성**(키-값 쌍).

### 문법: 만든 것을 바로 돌려받기
`CREATE` 문 **하나만** 적으면 돌려받는 값이 없습니다(만들기만 합니다). 방금 만든 것을 그 자리에서 확인하려면 노드에 **변수**를 붙이고 문장 끝에 **`RETURN`** 을 이어 씁니다.

```text
CREATE (p:Project {name: '알림봇', status: '진행'}) RETURN p.name AS name
```

이러면 다시 찾을 필요가 없습니다. 아래에서 **만들고 다시 찾는 두 걸음**을 먼저 해 본 뒤, 같은 일을 **한 문장**으로 끝내는 쪽과 견줘 봅니다.

노드를 만드는 `CREATE` 문 하나를 조각내 보면 이렇게 읽힙니다. 이 그림만 익혀 두면 오늘 나오는 모든 노드 패턴을 읽을 수 있습니다.

<img src="images/node-pattern-anatomy.png" width="820">

In [ ]:
# 새 직원 '한소희'(백엔드) 노드를 하나 만든다. 변수 없이 레이블과 속성만 적으면 노드가 하나 생긴다
made = run_cypher("CREATE (:Employee {name: '한소희', role: '백엔드'})")
# CREATE 만 적은 문장에는 RETURN 이 없다. 그래서 돌려받는 행도 없다(만들기는 됐다)
print("CREATE 만 적은 문장이 돌려준 행:", made)

돌려받은 것이 없으니 정말 들어갔는지 알 수 없습니다. **바로 다음 줄에서 찾아 확인**해 봅니다. 찾는 낱말이 `MATCH` 인데, 여기서는 "만든 것이 들어갔는지 보는 용도"로만 쓰고 문법은 2장에서 제대로 다룹니다.

In [ ]:
# 방금 만든 한소희를 그 자리에서 찾아 확인한다. 속성 map {name: '한소희'} 로 그 한 명만 집는다
# RETURN 의 별칭이 곧 결과 dict 의 키다
print(run_cypher("MATCH (e:Employee {name: '한소희'}) RETURN e.name AS name, e.role AS role"))

만들고 → 다시 찾고, 두 걸음이 들었습니다. 이 **다시 찾는 한 걸음을 없애는 것**이 위에서 본 `CREATE … RETURN` 입니다. 같은 일을 한 문장으로 해 봅니다.

In [ ]:
# 같은 CREATE 라도 노드에 변수 p 를 붙이고 RETURN 을 이어 쓰면 방금 만든 것을 그 자리에서 돌려받는다
# 알림봇 프로젝트를 새로 만들면서 이름·상태를 바로 확인한다. 다시 MATCH 할 필요가 없다
made = run_cypher(
    "CREATE (p:Project {name: '알림봇', status: '진행'}) RETURN p.name AS name, p.status AS status"
)
print("CREATE ... RETURN 이 돌려준 행:", made)

> 만들면서 돌려받을 필요가 없을 때는 `RETURN` 을 생략하면 됩니다. 다만 **변수를 붙이지 않으면** (`(:Project {...})`) 그 노드를 문장 안에서 가리킬 방법이 없어 `RETURN` 도 쓸 수 없습니다. 돌려받고 싶으면 변수부터 붙이세요.

> 속성 하나로 노드를 콕 집어 찾을 때는 패턴 안에 **속성 map** `{name: '한소희'}` 을 적습니다. 이게 Cypher 의 가장 기본적인 필터입니다(더 복잡한 조건은 다음 시간의 WHERE 로). `{role: '백엔드', years: 5}` 처럼 **둘 이상** 적으면 그 값이 **모두** 맞는 노드만 걸립니다.

### 🖐️ 함께 따라하기: 새 동아리 노드 만들기

여기서부터는 **동아리 그래프**(캠퍼스라운지)로 옮겨 같은 문법을 써 봅니다. 캠퍼스라운지에 **요리동아리**가 새로 생겼습니다. 레이블이 `Club`, 속성이 `name='요리동아리'` 인 노드를 만들되, 시연의 알림봇처럼 노드에 변수 `c` 를 붙이고 **`RETURN` 을 이어 써** 방금 만든 이름을 별칭 `name` 으로 그 자리에서 돌려받아 출력하세요(다시 `MATCH` 하지 않습니다).

**확인 기준**: 출력이 `[{'name': '요리동아리'}]` 한 줄이면 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 레이블 Club, 속성 {name: '요리동아리'} 인 노드를 CREATE 하되 노드에 변수 c 를 붙인다
# 2) 같은 문장 끝에 RETURN 을 이어 써 c.name 을 별칭 name 으로 돌려받는다
# 3) 돌려받은 결과를 출력한다

### ✅ 바로 확인 퀴즈

**1.** `CREATE (:Project {name: '알림봇', status: '진행'})` 이 만드는 것은 무엇인가요?

<details><summary>정답 보기</summary>

레이블이 **`Project`** 이고 속성이 `name='알림봇'`, `status='진행'` 인 **노드 하나**를 만듭니다.

</details>

**2.** 노드 패턴 `(:Employee {name: '한소희'})` 에서 `:Employee` 와 `{name: '한소희'}` 는 각각 무엇인가요?

<details><summary>정답 보기</summary>

`:Employee` 는 노드의 **레이블(종류)**, `{name: '한소희'}` 는 노드의 **속성**입니다.

</details>

**3.** `CREATE (:Team {name: '인프라팀'})` 과 `CREATE (t:Team {name: '인프라팀'}) RETURN t.name AS name` 은 그래프에 남는 결과가 어떻게 다른가요?

<details><summary>정답 보기</summary>

그래프에 남는 것은 **똑같습니다**(둘 다 인프라팀 노드 하나). 다른 것은 **돌려받는 값**뿐입니다. 앞은 빈 리스트를, 뒤는 `[{'name': '인프라팀'}]` 을 돌려줍니다. 돌려받으려면 노드에 **변수**(`t`)를 붙여야 `RETURN` 이 그 노드를 가리킬 수 있습니다.

</details>

---
## 1-2. 속성에는 자료형이 있다

### 왜 필요할까요?
1-1 에서 속성을 `{name: '한소희', years: 5}` 처럼 적었습니다. 이름은 따옴표를 씌웠고 숫자는 안 씌웠죠. 그냥 표기 습관이 아니라 **어떻게 적었느냐가 그 값의 자료형을 정합니다.** 그리고 자료형이 정해지면 **나중에 그 값으로 무엇을 할 수 있는지**가 함께 정해집니다.

### 문법: 여섯 가지 자료형
| 자료형 | 표기 예 | 비고 |
|---|---|---|
| 정수 | `years: 5` | 64비트 |
| 실수 | `rating: 4.5` | 정수와 구분됩니다 |
| 문자열 | `name: '한소희'` | 따옴표 필수. `years: '5'` 는 숫자가 아닙니다 |
| 불리언 | `active: true` | `true`·`false` 는 **소문자**. 파이썬의 `True` 와 다릅니다 |
| 리스트 | `tags: ['문서', '검색']` | 같은 자료형끼리만 담깁니다 |
| 날짜·시간 | `started: date('2024-05-01')` | `date`·`datetime`·`localdatetime`·`time`·`duration` |

오늘 실습에서 실제로 쓰는 것은 **문자열과 정수** 둘입니다. 나머지는 이런 것이 더 있다는 것만 알아 두고, 아래에서 한 번씩 넣어 봅니다.


표에 적은 여섯 자료형을 한 노드에 다 넣어 봅니다. 돌려받은 값이 파이썬에서 어떤 타입이 되는지도 함께 찍어 보면, Cypher 의 자료형과 파이썬의 자료형이 그대로 짝지어진다는 것이 보입니다.

In [ ]:
# 자료형마다 적는 법이 다르다. 문자열만 따옴표로 감싸고 숫자·불리언은 그대로 적는다
# 날짜는 date('2024-05-01') 처럼 함수로 만든다. 따옴표만 씌운 '2024-05-01' 과는 다른 값이다
made = run_cypher(
    "CREATE (p:Project {name: '사내위키', budget: 3000, rating: 4.5, active: true, "
    "tags: ['문서', '검색'], started: date('2024-05-01')}) "
    "RETURN p.name AS name, p.budget AS budget, p.rating AS rating, p.active AS active, "
    "p.tags AS tags, p.started AS started"
)
# 값과 함께 파이썬 타입 이름도 찍는다. Cypher 의 자료형이 그대로 건너온다
for key, value in made[0].items():
    print(f"{key:8} {value!r:28} {type(value).__name__}")

> 파이썬 쪽에서 `int`·`float`·`bool`·`list` 로 그대로 옵니다. 날짜만 `Date` 라는 전용 타입인데, 이게 바로 다음 셀에서 쓸모가 있습니다.

### 날짜를 날짜로 적으면 무엇이 달라지나
날짜는 글자로 적어도 사람 눈에는 같아 보입니다. 그런데 **날짜 값으로 적어 두면 연·월을 꺼내거나 두 날짜 사이를 잴 수 있습니다.** 글자는 그냥 글자라 그런 일을 못 합니다.

- `date()` : 오늘 날짜. 괄호를 비우면 실행하는 날이 들어갑니다.
- `날짜값.year`·`.month`·`.day` : 성분을 꺼냅니다.
- `duration.between(앞, 뒤)` : 두 날짜 사이의 기간. `.months`·`.days` 로 꺼냅니다.

In [ ]:
# 시작일에서 연·월을 꺼내고, 오늘까지 몇 달이 지났는지 센다
# date() 는 오늘 날짜라 아래 결과는 실행하는 날마다 달라진다
rows = run_cypher(
    "MATCH (p:Project {name: '사내위키'}) "
    "RETURN p.started AS started, p.started.year AS year, p.started.month AS month, "
    "date() AS today, duration.between(p.started, date()).months AS months"
)
print(rows[0])

In [ ]:
# 같은 '2024-05-01' 을 따옴표로만 적으면 날짜가 아니라 글자다
run_cypher("CREATE (:Project {name: '사내위키2', started: '2024-05-01'})")
row = run_cypher(
    "MATCH (a:Project {name: '사내위키'}), (b:Project {name: '사내위키2'}) "
    "RETURN a.started AS by_date, b.started AS by_quotes"
)[0]
print("date() 로 만든 값:", row['by_date'], type(row['by_date']).__name__)
print("따옴표로 적은 값 :", row['by_quotes'], type(row['by_quotes']).__name__)

> 겉보기는 같아도 **글자로 적은 쪽은 `.year` 를 꺼낼 수도, 기간을 잴 수도 없습니다.** 적재할 때 날짜를 글자로 넣어 두면 나중에 날짜로 다루려고 전부 고쳐야 합니다. 리스트도 마찬가지로 **같은 자료형끼리만** 담깁니다. `['a', 1]` 처럼 섞으면 저장에서 거부됩니다.

> 날짜로 **거르는** 조회("마감일이 이 날 이후인 프로젝트" 같은 것)는 조건을 다루는 낱말이 따로 필요해서 **다음 시간의 `WHERE`** 에서 이어 합니다.

### 🖐️ 함께 따라하기: 새 행사를 자료형 갖춰 만들기

다시 **동아리 그래프**입니다. 새 행사 **겨울음악회**를 등록하되 여섯 자료형을 골고루 써 보세요. `Event` 노드에 `name='겨울음악회'`(문자열), `fee=5000`(정수), `rating=4.2`(실수), `indoor=true`(불리언), `tags=['공연', '밴드']`(리스트), `opens=date('2026-12-05')`(날짜)를 적고, 노드에 변수 `v` 를 붙여 **같은 문장 끝에 `RETURN`** 으로 `fee`·`indoor`·`opens` 를 같은 이름의 별칭으로 돌려받아 출력하세요. 이어서 `opens` 에서 **연도만** 꺼내 별칭 `year` 로 조회해 출력합니다.

**확인 기준**: 첫 출력의 `fee` 는 `5000`, `indoor` 는 `True`, `opens` 는 `2026-12-05` 이고, 두 번째 출력의 `year` 는 `2026` 입니다. `opens` 를 따옴표로만 적었다면 연도를 꺼낼 수 없습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) Event 노드를 변수 v 와 함께 CREATE 하고 여섯 속성을 자료형에 맞게 적는다
# 2) 같은 문장 끝에 RETURN 을 이어 써 fee·indoor·opens 를 별칭 그대로 돌려받아 출력한다

In [ ]:
# 🖐️ 함께 따라하기 (이어서)
# 3) 겨울음악회를 MATCH 해 opens 에서 연도만 꺼내 별칭 year 로 조회해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `years: 5` 와 `years: '5'` 는 무엇이 다른가요?

<details><summary>정답 보기</summary>

앞은 **정수**, 뒤는 **문자열**입니다. 화면에 찍히는 모양은 비슷해 보여도 뒤쪽은 숫자가 아니라 글자라, 크기 비교나 계산에 쓸 수 없습니다.

</details>

**2.** 날짜를 `date('2024-05-01')` 이 아니라 `'2024-05-01'` 로 적으면 못 하게 되는 일 두 가지를 드세요.

<details><summary>정답 보기</summary>

1. `.year`·`.month` 처럼 **성분을 꺼내지** 못합니다. 글자에는 연·월이라는 것이 없습니다.
2. `duration.between(...)` 으로 **두 날짜 사이를 재지** 못합니다. 크기 비교도 성립하지 않습니다.

</details>

**3.** `tags: ['공연', 3]` 처럼 리스트에 문자열과 숫자를 섞으면 어떻게 되나요?

<details><summary>정답 보기</summary>

**저장에서 거부됩니다.** 리스트 속성은 **같은 자료형끼리만** 담깁니다. 섞어야 한다면 값을 문자열로 통일하거나, 아예 별도의 노드로 빼야 합니다.

</details>

---
## 1-3. 레이블은 여러 개일 수 있다

### 왜 필요할까요?
지금까지 노드마다 레이블을 하나씩 붙였습니다. 그런데 한 사람이 **직원이면서 사내 멘토**이기도 하다면요. 둘 중 하나를 고르라고 하면 나머지 한쪽 정보가 사라집니다. 한 노드가 두 종류에 동시에 속할 수 있어야 합니다.

### 문법: 콜론을 이어 붙인다
```text
CREATE (:Employee:Mentor {name: '류지호', role: '데이터'})
```

- 콜론을 **이어서** 적으면 됩니다. 순서는 상관없습니다(`:Mentor:Employee` 도 같습니다).
- 레이블 하나만 적어 찾아도 그 노드가 걸립니다.
- 나중에 `SET` 으로 더 붙이고 `REMOVE` 로 뗄 수도 있습니다(3장에서 합니다).

In [ ]:
# 신입 교육을 맡은 류지호는 직원이면서 멘토다. 콜론을 이어 붙여 레이블 둘을 함께 적는다
run_cypher("CREATE (:Employee:Mentor {name: '류지호', role: '데이터'})")
# 오세정은 회사 밖에서 초빙한 멘토다. 직원은 아니니 Mentor 레이블 하나만 붙는다
run_cypher("CREATE (:Mentor {name: '오세정', role: '커리어'})")
print("만들었습니다")   # CREATE 만 적었으니 돌려받는 행은 없다

In [ ]:
# 레이블을 하나만 적어 찾아도 걸린다. Employee 로 찾아도, Mentor 로 찾아도 같은 그 노드다
as_employee = run_cypher("MATCH (e:Employee {name: '류지호'}) RETURN e.role AS role")
as_mentor = run_cypher("MATCH (m:Mentor {name: '류지호'}) RETURN m.role AS role")
print("Employee 로 찾기:", as_employee, "/ Mentor 로 찾기:", as_mentor)

In [ ]:
# 패턴에 레이블 둘을 함께 적으면 '두 종류에 다 속하는' 노드만 걸린다(둘 중 하나가 아니다)
both = run_cypher("MATCH (x:Employee:Mentor) RETURN x.name AS name")
all_mentors = run_cypher("MATCH (m:Mentor) RETURN m.name AS name")
# 멘토는 둘인데 위 결과는 류지호 하나뿐이다. 오세정은 직원이 아니라 Employee 조건에서 걸러진다
print("Employee 이면서 Mentor:", sorted(r['name'] for r in both),
      "/ Mentor 전체:", sorted(r['name'] for r in all_mentors))

> 레이블을 여러 개 적은 패턴은 **AND** 로 읽습니다. `(x:Employee:Mentor)` 는 "Employee **이면서** Mentor" 이지 "Employee **이거나** Mentor" 가 아닙니다. 그래서 멘토가 둘인데도 류지호만 걸리고, 직원이 아닌 오세정은 빠집니다. 레이블은 나중에 `SET` 으로 더 붙이고 `REMOVE` 로 뗄 수도 있습니다(3장에서 합니다).

그럼 반대로 **"둘 중 하나라도"**(OR)는 어떻게 찾을까요. 한 패턴에 레이블 둘을 적는 것으로는 안 됩니다. 그건 방금 본 대로 AND 니까요. 지금 배운 것만으로 푸는 방법은 **각각 찾아 합치는** 것입니다.

In [ ]:
# 'Employee 이거나 Mentor' 는 한 패턴으로 못 적는다. 레이블을 나열하면 AND 가 되기 때문이다
# 그래서 각각 따로 찾은 뒤 파이썬에서 합친다
emp_rows = run_cypher("MATCH (e:Employee) RETURN e.name AS name")
mentor_rows = run_cypher("MATCH (m:Mentor) RETURN m.name AS name")
# 류지호는 양쪽에 다 걸린다. 집합(중괄호)으로 합치면 겹치는 이름이 한 번만 남는다
either = {r['name'] for r in emp_rows} | {r['name'] for r in mentor_rows}
print("Employee 이거나 Mentor:", sorted(either))

> 같은 일을 **Cypher 문장 하나로** 끝내는 방법도 있습니다. 조건을 `이거나` 로 묶는 `OR` 인데, 그건 조건을 다루는 낱말이라 **다음 시간의 `WHERE`** 에서 배웁니다. 지금 기억할 것은 하나입니다. **패턴에 레이블을 나란히 적는 것은 언제나 AND 다.**

### 🖐️ 함께 따라하기: 학생이면서 운영진

다시 **동아리 그래프**입니다. **차은우**는 학생이면서 동아리 **운영진**입니다. 레이블 두 개 (`Student` 와 `Staff`)를 함께 붙여 `name='차은우'`, `grade=2` 인 노드를 만드세요. 그런 다음 **두 레이블을 다 가진** 노드의 이름을 별칭 `name` 으로 조회해 정렬 출력하고, 이어서 `Student` 레이블만으로 세었을 때의 학생 수도 출력해 견줘 보세요.

**확인 기준**: 앞 결과는 `['차은우']` 하나뿐이고, 뒤의 학생 수는 시드 5명에 차은우가 더해져 **6** 입니다. 차은우는 `Student` 로도 `Staff` 로도 걸리는 **같은 노드 하나**입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 레이블 Student 와 Staff 를 콜론으로 이어 붙여 차은우 노드를 CREATE 한다
# 2) 두 레이블을 함께 적은 패턴으로 MATCH 해 이름을 별칭 name 으로 조회해 정렬 출력한다

In [ ]:
# 🖐️ 함께 따라하기 (이어서)
# 3) Student 레이블만으로 학생 수를 세어 출력한다(차은우도 학생으로 걸린다)

### ✅ 바로 확인 퀴즈

**1.** `MATCH (x:Employee:Mentor)` 는 어떤 노드를 찾나요?

<details><summary>정답 보기</summary>

**두 레이블을 다 가진** 노드만 찾습니다. 레이블을 나란히 적은 것은 **AND** 이지 OR 이 아닙니다. 그래서 멘토이기만 한 오세정은 걸리지 않습니다.

</details>

**2.** "직원이거나 멘토인 사람" 을 지금 배운 것만으로 찾으려면 어떻게 하나요?

<details><summary>정답 보기</summary>

**각각 따로 찾아 파이썬에서 합칩니다.** 한 패턴에 레이블 둘을 적는 것으로는 안 됩니다(그건 AND 니까요). Cypher 문장 하나로 끝내는 `OR` 은 다음 시간의 `WHERE` 에서 배웁니다.

</details>

**3.** 류지호에게 `Mentor` 레이블을 하나 더 붙였습니다. 이제 `MATCH (e:Employee)` 로 찾으면 류지호가 나올까요?

<details><summary>정답 보기</summary>

**나옵니다.** 레이블을 더 붙인 것이지 `Employee` 를 뗀 것이 아닙니다. 레이블 하나만 적은 패턴은 그 레이블을 **가지고 있기만 하면** 걸립니다.

</details>

---
## 1-4. 관계 만들기

### 왜 필요할까요?
노드만 있으면 흩어진 점일 뿐입니다. **한소희가 개발팀에 속한다**는 사실은 두 노드를 잇는 **관계**로 담습니다.

### 문법: 관계 패턴
관계는 두 노드 사이의 **화살표**로 그립니다.

```text
(a)-[:WORKS_IN]->(b)
```

- `-[:WORKS_IN]->` : **방향 있는 관계**. 종류는 `WORKS_IN`, 화살표는 `a` 에서 `b` 로 향합니다.
- 관계 종류는 **대문자 스네이크**(`WORKS_IN`)로 쓰는 게 관례입니다.

이미 있는 두 노드를 잇는 관계를 만들 때는, 먼저 **MATCH 로 두 노드를 찾아** 변수(`a`, `b`)에 담고, 그 변수 사이에 `CREATE` 로 관계를 긋습니다.

관계 부분도 조각내 보면 이렇게 읽힙니다. 1-1 의 노드 해부도와 이어 붙이면 문장 하나가 완성됩니다.

<img src="images/rel-pattern-anatomy.png" width="820">

In [ ]:
# 한소희(직원)를 개발팀(팀)에 소속시킨다. 두 노드를 MATCH 로 찾아 관계를 CREATE
# MATCH 의 쉼표는 "둘을 각각 찾아라". 그렇게 e·t 에 담긴 두 노드 사이에 화살표를 하나 긋는다
run_cypher(
    "MATCH (e:Employee {name: '한소희'}), (t:Team {name: '개발팀'}) "
    "CREATE (e)-[:WORKS_IN]->(t)"
)
# 이번 MATCH 는 노드 하나가 아니라 "한소희에서 팀으로 나가는 WORKS_IN" 이라는 모양 전체를 찾는다
print(run_cypher(
    "MATCH (e:Employee {name: '한소희'})-[:WORKS_IN]->(t:Team) RETURN t.name AS team"
))

> `MATCH (e), (t) CREATE (e)-[:...]->(t)` 는 **찾아서 잇는다**는 한 문장입니다. 쉼표로 노드 둘을 각각 찾고, 그 사이에 화살표를 긋습니다.

### 🖐️ 함께 따라하기: 학생을 행사에 참가시키기

다시 **동아리 그래프**입니다. **서지안**을 **출사모임** 행사에 참가시키세요. 관계 종류는 `JOINS`, 방향은 학생에서 행사로 향합니다. 서지안(`Student`)과 출사모임(`Event`)을 각각 MATCH 로 찾아 그 사이에 `JOINS` 관계를 CREATE 한 뒤, 서지안의 참가 관계를 다시 MATCH 해 행사 이름을 별칭 `event` 로 RETURN 하고 출력해 확인하세요.

**확인 기준**: 행사가 **두 개** 나옵니다(봄전시회·출사모임). 서지안은 시드에서 이미 봄전시회에 참가 중이라, 방금 그은 화살표까지 둘이 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 서지안(Student)과 출사모임(Event)을 MATCH 로 각각 찾는다(쉼표로 나열)
# 2) 그 사이에 (s)-[:JOINS]->(v) 관계를 CREATE 한다
# 3) 참가 관계를 MATCH 해 v.name 을 별칭 event 로 RETURN 하고 출력한다

### ✅ 바로 확인 퀴즈

**1.** 패턴 `(e)-[:WORKS_IN]->(t)` 에서 화살표의 방향이 뜻하는 것은?

<details><summary>정답 보기</summary>

관계가 **`e`(직원)에서 `t`(팀)로** 향한다는 뜻입니다. "직원이 팀에 속한다"를 그대로 그린 것입니다.

</details>

**2.** 이미 있는 두 노드를 잇는 관계를 만들 때, `CREATE` 앞에 **`MATCH`** 를 먼저 쓰는 이유는?

<details><summary>정답 보기</summary>

관계를 그으려면 **양쪽 노드를 먼저 찾아 변수에 담아야** 하기 때문입니다. MATCH 로 두 노드를 잡고 그 변수 사이에 화살표를 긋습니다.

</details>

---
## 1-5. 관계에 속성 담기

### 왜 필요할까요?
"한소희가 앱개편에 배정됐다"까지는 화살표로 담았습니다. 그런데 **언제부터** 배정됐고 **주당 몇 시간**을 쓰는지는 어디에 둘까요. 사람의 값도 아니고 프로젝트의 값도 아닙니다. **그 배정 하나에만 해당하는 값**이니 화살표 자신이 들고 있어야 합니다. 지난 시간에 본 Movies 그래프의 `roles`(배역)·`rating`(평점)이 바로 이런 값이었습니다.

### 문법: 관계에도 속성을 붙인다
```text
(e)-[:ASSIGNED_TO {since: 2024, hours: 20}]->(p)
```

- 노드와 똑같이 `{키: 값}` 을 적습니다. 자리만 소괄호 안이 아니라 **대괄호 안**입니다.
- 관계 속성을 꺼내려면 관계를 **변수에 담습니다**: `-[r:ASSIGNED_TO]->`. 그러면 노드 속성을 `e.name` 으로 읽듯 관계 속성을 **`r.since`** 로 읽습니다.

1-4 해부도의 대괄호 안에 속성이 하나 더 들어가는 셈입니다. 노드 속성과 관계 속성이 각각 어디에 붙는지 나란히 놓고 보면 이렇습니다.

<img src="images/rel-property.png" width="820">

In [ ]:
# 배정 시점(since)·주당 시간(hours)은 사람의 값도 프로젝트의 값도 아닌 '이 배정'의 값이다
# 그래서 화살표 안 {} 에 적는다
# 관계에 변수 r 을 붙였으므로 같은 문장의 RETURN 으로 그 값을 돌려받을 수 있다
made = run_cypher(
    "MATCH (e:Employee {name: '한소희'}), (p:Project {name: '앱개편'}) "
    "CREATE (e)-[r:ASSIGNED_TO {since: 2024, hours: 20}]->(p) "
    "RETURN r.since AS since, r.hours AS hours"
)
print("방금 만든 배정 관계:", made)

In [ ]:
# 이미 있는 관계의 속성을 읽을 때도 관계에 변수를 붙인다. -[r:ASSIGNED_TO]-> 의 r 이 그 화살표다
# 노드 값은 p.name 으로, 관계 값은 r.since 로. 꺼내는 방식은 똑같고 어디에 붙어 있느냐만 다르다
rows = run_cypher(
    "MATCH (e:Employee {name: '한소희'})-[r:ASSIGNED_TO]->(p:Project) "
    "RETURN p.name AS project, r.since AS since, r.hours AS hours"
)
print(rows)

> 시드 셀이 만든 다른 배정 관계에는 `since`·`hours` 가 없습니다. 관계 속성은 **관계마다 있을 수도 없을 수도** 있고, 없는 값을 꺼내면 파이썬에서 `None` 으로 옵니다(노드 속성도 마찬가지입니다).

### 🖐️ 함께 따라하기: 좌석을 참가 관계에 담기

다시 **동아리 그래프**입니다. **노태윤**이 **봄전시회**에도 참가하기로 했고, 배정된 좌석은 `'A12'` 입니다. 좌석은 학생의 값도 행사의 값도 아니라 **그 참가 하나의 값**이죠. 노태윤(`Student`)과 봄전시회(`Event`)를 각각 `MATCH` 로 찾아 그 사이에 `JOINS` 관계를 `CREATE` 하되, 관계에 변수 `r` 과 속성 `{seat: 'A12'}` 를 함께 적고, 같은 문장 끝에 `RETURN` 을 이어 써 `r.seat` 을 별칭 `seat` 으로 돌려받아 출력하세요.

**확인 기준**: 출력이 `[{'seat': 'A12'}]` 한 줄이면 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 노태윤(Student)과 봄전시회(Event)를 쉼표로 나란히 MATCH 한다
# 2) 그 사이에 관계를 CREATE 하되 대괄호 안에 변수 r 과 {seat: 'A12'} 를 함께 적는다
# 3) 같은 문장 끝에 RETURN 을 이어 써 r.seat 을 별칭 seat 으로 돌려받아 출력한다

### ✅ 바로 확인 퀴즈

**1.** 직원이 프로젝트에 배정된 **날짜**는 직원 노드·프로젝트 노드·배정 관계 중 어디에 두어야 하나요?

<details><summary>정답 보기</summary>

**배정 관계**에 둡니다. 같은 사람이 여러 프로젝트에 배정될 수 있고 그때마다 날짜가 다르므로, 그 값은 사람의 값도 프로젝트의 값도 아니라 **그 배정 하나의 값**입니다.

</details>

**2.** `MATCH (e:Employee)-[:ASSIGNED_TO]->(p:Project) RETURN r.since` 는 왜 안 될까요?

<details><summary>정답 보기</summary>

관계에 **변수를 붙이지 않았기** 때문입니다. `r` 이라는 이름이 어디에도 없으니 `r.since` 를 읽을 수 없습니다. `-[r:ASSIGNED_TO]->` 처럼 대괄호 안에 변수를 먼저 붙여야 합니다.

</details>

---
## 1-6. 노드와 관계를 한 문장으로

### 왜 필요할까요?
1-4 에서는 관계를 그으려고 `MATCH` 로 양쪽 노드를 먼저 찾았습니다. 그런데 **양쪽 다 아직 없는** 노드라면 찾을 것이 없죠. 그럴 때는 화살표 양 끝에 노드 패턴을 그대로 그려 한 문장에 다 만듭니다.

### 문법: 노드와 관계를 한 문장으로
```text
CREATE (e:Employee {name: '문가영', role: '기획'})-[:WORKS_IN]->(t:Team {name: '기획팀'})
```

- 화살표 양 끝에 노드 패턴을 그대로 적으면 **한 문장에 셋을 다** 만듭니다(노드 2개 + 관계 1개).
- 화살표를 더 이어 길게 그려도 됩니다: `CREATE (a)-[:R1]->(b)-[:R2]->(c)`.

In [ ]:
# 기획팀도 문가영도 아직 없다. 그러니 MATCH 없이 한 문장에 노드 둘과 관계 하나를 함께 만든다
made = run_cypher(
    "CREATE (e:Employee {name: '문가영', role: '기획'})-[:WORKS_IN]->(t:Team {name: '기획팀'}) "
    "RETURN e.name AS name, t.name AS team"
)
print("한 문장으로 만든 직원과 팀:", made)

> ⚠️ **한쪽이 이미 있는 노드라면 이 형태를 쓰면 안 됩니다.** `CREATE` 는 적힌 패턴을 **통째로 새로** 만듭니다. 개발팀이 이미 있는데 `CREATE (:Employee {name: '...'})-[:WORKS_IN]->(:Team {name: '개발팀'})` 라고 쓰면, 이름만 같은 **개발팀 노드가 하나 더** 생기고 새 직원은 그 새 개발팀에 붙습니다. 겉보기에는 잘된 것 같은데 조회 결과가 둘로 갈라지죠. 그래서 한 문장 생성은 **양쪽이 모두 새것일 때만** 쓰고, 한쪽이 이미 있으면 1-4 처럼 `MATCH` 로 찾아 잇습니다. 다음 시간에 배울 **MERGE** 가 이 고민 자체를 없애 줍니다.

같은 상황을 두 가지로 써 보면 결과가 이렇게 갈립니다.

<img src="images/create-duplicate-trap.png" width="820">

### 🖐️ 함께 따라하기: 새 학생과 새 동아리를 한 문장으로

다시 **동아리 그래프**입니다. 신입생 **하지원**(`Student`, `major='음악'`, `grade=1`)이 새로 만들어진 **노래동아리**(`Club`)의 창립 멤버입니다. 둘 다 아직 그래프에 없으니 **한 문장**으로 학생 노드·동아리 노드·`BELONGS_TO` 관계를 함께 만드세요. 관계에는 가입 연도 `{joined: 2026}` 을 붙이고, 같은 문장 끝에 `RETURN` 을 이어 써 학생 이름을 별칭 `name`, 동아리 이름을 별칭 `club`, 가입 연도를 별칭 `joined` 로 돌려받아 출력하세요.

**확인 기준**: 출력이 `[{'name': '하지원', 'club': '노래동아리', 'joined': 2026}]` 한 줄이면 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 학생 노드에 변수 s, 동아리 노드에 변수 c, 관계에 변수 r 을 붙여 한 문장으로 CREATE 한다
# 2) 관계에는 {joined: 2026} 속성을 붙인다(대괄호 안에 적는다)
# 3) 같은 문장 끝에 RETURN 을 이어 써 세 값을 별칭 name·club·joined 로 돌려받아 출력한다

### ✅ 바로 확인 퀴즈

**1.** 개발팀이 이미 있는데 `CREATE (:Employee {name: '나단'})-[:WORKS_IN]->(:Team {name: '개발팀'})` 을 실행하면 어떻게 되나요?

<details><summary>정답 보기</summary>

이름만 같은 **개발팀 노드가 하나 더** 생기고, 나단은 원래 개발팀이 아니라 그 새 노드에 이어집니다. `CREATE` 는 이미 있는지 보지 않고 적힌 패턴을 통째로 새로 만들기 때문입니다. 이럴 때는 개발팀을 `MATCH` 로 먼저 찾아야 합니다.

</details>

**2.** 신입 사원 두 명이 **새로 만든 팀**에 함께 들어옵니다. 셋 다 그래프에 없을 때, `MATCH` 를 쓰지 않고 한 문장 생성만으로 처리하면 무엇이 문제일까요?

<details><summary>정답 보기</summary>

첫 번째 사원을 만들 때 팀도 함께 생기고, 두 번째 사원을 같은 방식으로 만들면 **같은 이름의 팀이 하나 더** 생깁니다. 팀은 첫 문장에서 한 번만 만들고, 두 번째 사원은 그 팀을 `MATCH` 로 찾아 이어야 합니다.

</details>

---
# 2. 조회: 찾아서 돌려받기

만든 데이터를 다시 꺼내는 일은 두 걸음입니다. **어떤 모양을 찾을지**(`MATCH`)와 **그중 무엇을 돌려받을지**(`RETURN`)를 정합니다. 2-1 에서 그 기본형을, 2-2 에서 돌려받는 값의 모양을 고르는 법을 봅니다.

## 2-1. MATCH 와 RETURN

### 왜 필요할까요?
만든 데이터를 **다시 꺼내 보는** 것이 쿼리의 핵심입니다. `MATCH` 로 원하는 모양을 찾고, `RETURN` 으로 그중 무엇을 돌려받을지 고릅니다.

### 문법: MATCH … RETURN … AS
```text
MATCH (e:Employee) RETURN e.name AS name
```

- `MATCH (e:Employee)` : 레이블이 `Employee` 인 노드를 **모두** 찾아 변수 `e` 에 담습니다.
- `RETURN e.name` : 그중 **`name` 속성**만 돌려받습니다.
- `AS name` : 돌려받는 값의 **이름(별칭)**을 `name` 으로 정합니다. `run_cypher` 결과 dict 의 **키**가 이 별칭이 됩니다(`row['name']`).

찾는 범위를 좁히려면 패턴에 **레이블**이나 **속성 map**을 더 적습니다.

MATCH 는 그래프 전체에서 **패턴과 같은 모양**을 찾아냅니다. 아래 그림에서 진한 부분이 패턴에 걸린 자리이고, 그중 `RETURN` 에 적은 값만 돌려받습니다.

<img src="images/match-highlight.png" width="820">

In [ ]:
# 속성 map 없이 레이블만 적었으므로 Employee 노드를 전부 가져온다
rows = run_cypher("MATCH (e:Employee) RETURN e.name AS name")
print(rows)   # 별칭 name 이 키인 dict 의 리스트로 온다(순서는 정해져 있지 않다)

In [ ]:
# 속성 map 으로 범위를 좁힌다. 도착 노드에 변수를 안 붙인 (:Team {name: '개발팀'}) 은
# 조건으로만 쓰고 값은 돌려받지 않겠다는 뜻이다
rows = run_cypher(
    "MATCH (e:Employee)-[:WORKS_IN]->(:Team {name: '개발팀'}) RETURN e.name AS name"
)
print(rows)   # 개발팀 소속만 걸린다. 앞 절에서 넣은 한소희도 들어 있다

> `RETURN` 은 여러 값을 동시에 돌려받을 수도 있습니다: `RETURN e.name AS name, e.role AS role`. 그러면 결과 dict 에 키가 두 개(`name`, `role`) 생깁니다.

In [ ]:
# RETURN 에 값을 두 개 적으면 결과 dict 의 키도 두 개가 된다(별칭이 곧 키)
rows = run_cypher("MATCH (e:Employee) RETURN e.name AS name, e.role AS role")
print(rows)   # dict 마다 키가 name·role 두 개다. 값을 꺼낼 때는 row['name']·row['role']

### 🖐️ 함께 따라하기: 행사의 이름과 상태를 함께 조회

다시 **동아리 그래프**입니다. 모든 `Event` 노드의 **이름과 상태**를 한 번에 조회하세요. `name` 은 별칭 `name` 으로, `status` 는 별칭 `status` 로 RETURN 합니다. 결과를 출력해 각 dict 에 `name`·`status` 두 키가 있는지 확인하세요.

**확인 기준**: 행사 **네 개**가 나오고, 각 dict 에 `name`·`status` 두 키가 있습니다. 순서는 정해져 있지 않습니다. 1-2 에서 만든 **겨울음악회는 `status` 가 `None`** 으로 나오는데, 그때 그 속성을 적지 않았기 때문입니다. **없는 속성을 읽으면 에러가 아니라 널**이라는 것을 여기서 미리 봅니다(3-2 에서 다시 다룹니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 Event 노드를 모두 찾는다
# 2) v.name 을 별칭 name 으로, v.status 를 별칭 status 로 RETURN 한다
# 3) 결과를 출력한다

### ✅ 바로 확인 퀴즈

**1.** `MATCH (e:Employee) RETURN e.name AS name` 의 결과에서, 각 행의 이름 값을 파이썬으로 꺼내려면 어떤 키를 써야 하나요?

<details><summary>정답 보기</summary>

별칭이 `name` 이므로 **`row['name']`** 으로 꺼냅니다. `AS name` 이 결과 dict 의 키를 정합니다.

</details>

**2.** `MATCH (e:Employee {role: '백엔드'})` 처럼 패턴 안에 속성 map 을 적으면 어떻게 되나요?

<details><summary>정답 보기</summary>

그 속성과 **값이 일치하는 노드만** 찾습니다. 여기서는 역할이 `백엔드` 인 직원만 걸러집니다.

</details>

---
## 2-2. 돌려받는 모양 고르기

### 왜 필요할까요?
같은 노드를 찾아도 `RETURN` 에 무엇을 적느냐에 따라 파이썬에 오는 값의 **모양**이 달라집니다. 속성 하나를 골라 받을 수도, 노드를 통째로 받을 수도 있죠. 어느 쪽이 편한지는 그때그때 다릅니다.

속성을 골라 받을 때와 노드를 통째로 받을 때, 파이썬에 오는 모양을 나란히 보면 이렇습니다.

<img src="images/return-shapes.png" width="820">

### 속성 말고 노드를 통째로 돌려받으면
지금까지는 `e.name` 처럼 **속성 하나**를 골라 돌려받았습니다. `RETURN e` 라고 변수만 적으면 **노드 전체**가 옵니다. 파이썬에서는 dict 안에 dict 가 들어온 모양이 됩니다: `{'e': {'role': '백엔드', 'name': '한소희'}}`. 그래서 이름을 꺼내려면 `row['e']['name']` 처럼 두 번 들어가야 합니다. 안쪽 dict 의 **속성 순서는 정해져 있지 않으니** 순서에 기대지 마세요.

필요한 값이 정해져 있으면 **속성을 골라 별칭으로 받는 쪽**이 읽기 쉽습니다. 노드를 통째로 받는 것은 어떤 속성이 들어 있는지 **한 번 훑어볼 때** 편합니다.

In [ ]:
# RETURN 에 변수 이름만 적으면 그 노드가 통째로 온다. 별칭을 안 붙였으니 키는 변수 이름 그대로 'e'
rows = run_cypher("MATCH (e:Employee {name: '한소희'}) RETURN e")
print(rows[0])   # e 키 하나에 노드 전체가 dict 로 들어 있다(속성 순서는 그때그때 다르다)

In [ ]:
# 그래서 값 하나를 꺼내려면 두 번 들어간다. 노드를 통째로 받았을 때만 이렇게 쓴다
print(rows[0]['e']['name'])

### 쿼리가 길어지면 여러 줄로
쿼리는 한 줄에 다 적지 않아도 됩니다. 파이썬 **삼중따옴표**(`"""`) 문자열에 담아 **절마다 줄을 바꿔** 쓰면 훨씬 읽기 좋습니다. 뒤 단원의 긴 쿼리는 모두 이 모양으로 나옵니다.

쿼리 안에 설명을 남기고 싶으면 **`//`** 로 시작하는 줄을 넣습니다(파이썬의 `#` 에 해당하는 Cypher 주석입니다).

In [ ]:
# 삼중따옴표 문자열은 줄바꿈을 그대로 담는다. 절마다 줄을 바꿔 쓰면 어디까지가 MATCH 이고
# 어디부터가 RETURN 인지 눈에 바로 들어온다. 실행 결과는 한 줄로 적었을 때와 똑같다
rows = run_cypher("""
// 개발팀에 속한 직원의 이름과 역할
MATCH (e:Employee)-[:WORKS_IN]->(:Team {name: '개발팀'})
RETURN e.name AS name, e.role AS role
""")
print(sorted(r['name'] for r in rows))

### 🖐️ 함께 따라하기: 동아리 노드를 통째로 받아 보기

다시 **동아리 그래프**입니다. 이름이 `'사진동아리'` 인 `Club` 노드를 **통째로** 돌려받으세요(`RETURN c`). 결과를 변수에 담아 첫 행을 그대로 출력하고, 이어서 그 안에서 동아리 이름만 `row['c']['name']` 처럼 두 번 들어가 꺼내 출력하세요. 쿼리는 **삼중따옴표로 여러 줄**에 나눠 쓰고, 맨 위에 `//` 주석으로 무엇을 찾는 쿼리인지 한 줄 적어 보세요.

**확인 기준**: 첫 출력은 `c` 키 하나에 dict 가 들어 있고, 두 번째 출력은 `사진동아리` 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 삼중따옴표 쿼리에 // 주석 한 줄을 적고, 사진동아리 Club 노드를 MATCH 해 RETURN c 한다
# 2) 결과를 변수 rows 에 담고 첫 행을 그대로 출력한다

In [ ]:
# 🖐️ 함께 따라하기 (이어서)
# 3) 노드를 통째로 받았으니 이름을 꺼내려면 두 번 들어간다

### ✅ 바로 확인 퀴즈

**1.** `RETURN e.name AS name` 과 `RETURN e` 는 파이썬에서 값을 꺼내는 방법이 어떻게 다른가요?

<details><summary>정답 보기</summary>

앞은 별칭이 곧 키라 **`row['name']`** 한 번이면 됩니다. 뒤는 노드가 통째로 오므로 **`row['e']['name']`** 처럼 두 번 들어가야 합니다.

</details>

**2.** 쿼리를 삼중따옴표로 여러 줄에 나눠 쓰면 실행 결과가 달라지나요?

<details><summary>정답 보기</summary>

달라지지 않습니다. Cypher 는 줄바꿈을 빈칸과 똑같이 봅니다. 바뀌는 것은 **사람이 읽기 쉬운 정도**뿐이고, `//` 로 시작하는 줄은 주석이라 실행에서 무시됩니다.

</details>

---
# 3. 수정: 만든 것을 고치기

한 번 넣고 그대로 두는 데이터는 없습니다. 근속연수가 한 해 늘고, 없던 연락처가 생기고, 잘못 넣은 값은 도로 빼야 합니다. 그렇다고 노드를 지우고 다시 만들 수는 없습니다. **붙어 있던 화살표까지 함께 사라져** 소속도 배정도 전부 다시 그어야 하니까요. 그래서 노드는 그대로 두고 **그 안의 값만** 손보는 낱말이 둘 있습니다. 값을 넣는 **`SET`**(3-1)과, 값이 든 칸 자체를 빼는 **`REMOVE`**(3-2)입니다.

## 3-1. SET 으로 값 고치기·더하기

### 왜 필요할까요?
김서준이 한 해를 더 채워 근속연수가 5년에서 6년이 됐습니다. 바꿀 것은 **숫자 하나**뿐인데, 노드를 지우고 새로 만들면 개발팀 소속과 결제시스템 배정까지 다시 이어야 합니다. 값 하나만 갈아 끼우는 문법이 따로 있는 이유입니다.

### 문법: MATCH 로 찾아 SET
```text
MATCH (e:Employee {name: '김서준'}) SET e.years = 6
```

- **먼저 `MATCH` 로 찾아 변수에 담습니다.** `SET` 은 그 변수가 가리키는 것의 값을 고칩니다.
- 적은 속성이 **이미 있으면 덮어쓰고, 없으면 새로 만들어 붙입니다.** '속성을 더하는' 문법이 따로 있지 않습니다. 둘 다 `SET` 한 낱말입니다.
- 고칠 값이 여럿이면 **쉼표로 잇습니다**: `SET e.role = '백엔드 리드', e.level = '시니어'`.
- 관계 속성도 같습니다. 관계에 변수를 붙여(`-[r:ASSIGNED_TO]->`) `SET r.hours = 30` 이라고 적습니다.
- 레이블을 붙일 때도 `SET` 을 씁니다. 이때는 점이 아니라 **콜론**입니다: `SET e:Manager` (붙인 레이블을 떼는 것은 3-2 에서 합니다).

`CREATE` 처럼 `SET` 뒤에도 **`RETURN` 을 이어 쓸 수 있습니다.** 그러면 **고친 뒤의 값**이 그 자리에서 돌아와 다시 `MATCH` 하지 않아도 됩니다.

같은 `SET` 한 낱말이 상황에 따라 두 가지로 동작합니다. 있던 칸이면 값을 갈아 끼우고, 없던 칸이면 칸을 새로 만들어 넣습니다.

<img src="images/set-property.png" width="820">

In [ ]:
# 김서준의 근속연수를 5 에서 6 으로 고친다. 이미 있는 속성이라 값이 덮어써진다
# SET 뒤에 RETURN 을 이어 쓰면 고친 뒤의 값이 그 자리에서 온다(고치기 전 값이 아니다)
after = run_cypher(
    "MATCH (e:Employee {name: '김서준'}) SET e.years = 6 RETURN e.years AS years"
)
print("고친 뒤 근속연수:", after)

In [ ]:
# 시드에 phone 속성은 없었다. 없는 속성에 SET 하면 에러가 아니라 그 칸이 새로 생긴다
# 즉 '값 고치기'와 '속성 더하기'가 같은 문장이다. 이 phone 은 3-2 에서 다시 떼어 낸다
after = run_cypher(
    "MATCH (e:Employee {name: '김서준'}) SET e.phone = '010-1234-5678' RETURN e.phone AS phone"
)
print("새로 붙인 연락처:", after)

In [ ]:
# 고칠 값이 여럿이면 SET 을 여러 번 쓰지 말고 쉼표로 잇는다. MATCH 를 한 번만 돌면 되기 때문이다
# 있던 role 은 덮어써지고 없던 level 은 새로 생긴다. 한 문장 안에서 둘이 함께 일어난다
after = run_cypher(
    "MATCH (e:Employee {name: '김서준'}) "
    "SET e.role = '백엔드 리드', e.level = '시니어' "
    "RETURN e.role AS role, e.level AS level"
)
print("한 문장으로 고친 두 값:", after)

> 관계 속성도 고치는 법이 같습니다. 다만 고칠 대상이 **화살표**이므로 1-5 에서처럼 관계 쪽에 변수를 붙여야 `SET` 이 그것을 가리킬 수 있습니다.

In [ ]:
# 김서준의 결제시스템 배정에 주당 시간을 적어 둔다. 시드가 만든 이 관계에는 hours 가 없었다
# 고칠 대상이 화살표라 관계 쪽에 변수 r 을 붙인다. 이게 없으면 SET 이 무엇을 고칠지 알 수 없다
after = run_cypher(
    "MATCH (:Employee {name: '김서준'})-[r:ASSIGNED_TO]->(:Project {name: '결제시스템'}) "
    "SET r.hours = 30 "
    "RETURN r.hours AS hours"
)
print("배정 관계에 담은 주당 시간:", after)

> ⚠️ **`SET` 이 고치는 범위는 앞의 `MATCH` 가 정합니다.** `MATCH (e:Employee) SET e.years = 0` 처럼 조건을 빠뜨리면 걸린 **직원 전원**의 근속연수가 0 이 됩니다. 한 명만 고칠 생각이었어도 데이터베이스는 걸린 것을 전부 고칩니다. 고치기 전에 같은 `MATCH` 를 **`RETURN` 으로 먼저 돌려** 몇 명이 걸리는지 눈으로 확인하는 습관을 들이세요.

### 🖐️ 함께 따라하기: 학년을 올리고 이메일을 새로 넣기

여기서부터는 다시 **동아리 그래프**(캠퍼스라운지)입니다. **서지안**이 2학년으로 올라갔고, 동아리 연락용 이메일 `'jian@campus.ac.kr'` 도 새로 등록했습니다. 서지안(`Student`)을 `MATCH` 로 찾아 **한 문장에서** `grade` 는 `2` 로 고치고 `email` 은 새로 붙이세요(두 속성을 **쉼표로** 잇습니다). 같은 문장 끝에 `RETURN` 을 이어 써 별칭 `grade`·`email` 로 돌려받아 출력하세요.

**확인 기준**: 출력이 `[{'grade': 2, 'email': 'jian@campus.ac.kr'}]` 한 줄이면 됩니다. `grade` 는 원래 있던 값을 덮어쓴 것이고, `email` 은 없던 칸이 새로 생긴 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 서지안(Student)을 속성 map 으로 MATCH 해 변수 s 에 담는다
# 2) SET 한 번에 s.grade 는 2 로 고치고 s.email 은 새로 붙인다(둘 사이를 쉼표로 잇는다)
# 3) 같은 문장 끝에 RETURN 을 이어 써 별칭 grade·email 로 돌려받아 출력한다

### ✅ 바로 확인 퀴즈

**1.** 노드에 원래 없던 속성을 `SET` 하면 어떻게 되나요?

<details><summary>정답 보기</summary>

그 속성이 **새로 생깁니다**. 에러가 나지 않습니다. Cypher 에는 '속성 추가' 문법이 따로 없고, **있으면 덮어쓰고 없으면 만드는** `SET` 한 낱말이 두 일을 다 합니다.

</details>

**2.** 값 하나를 바꾸려고 노드를 지웠다가 다시 `CREATE` 하면 무엇이 문제일까요?

<details><summary>정답 보기</summary>

그 노드에 **붙어 있던 관계가 함께 사라집니다.** 새로 만든 노드는 이름만 같을 뿐 화살표가 하나도 없는 다른 노드라, 소속·배정을 전부 다시 그어야 합니다. 값만 바꿀 때는 `SET` 을 씁니다.

</details>

**3.** `MATCH (e:Employee) SET e.years = 0` 을 실행하면 몇 명의 근속연수가 바뀌나요?

<details><summary>정답 보기</summary>

**걸린 직원 전원**입니다. `SET` 이 고치는 범위는 앞의 `MATCH` 가 정하는데, 여기에는 속성 map 도 없어 `Employee` 노드가 전부 걸립니다. 한 명만 고치려면 `(e:Employee {name: '김서준'})` 처럼 대상을 좁혀야 합니다.

</details>

---
## 3-2. REMOVE 로 속성·레이블 지우기

### 왜 필요할까요?
연락처를 더 이상 관리하지 않기로 했습니다. 빈 문자열 `''` 을 넣어 두면 값이 없는 것인지 빈 값인 것인지 나중에 알 수 없습니다. 이럴 때는 값을 비우는 게 아니라 **칸 자체를 떼어 냅니다.** 잘못 붙인 레이블도 마찬가지입니다.

### 문법: REMOVE 로 칸을 떼어 낸다
```text
MATCH (e:Employee {name: '김서준'}) REMOVE e.phone      // 속성 한 칸
MATCH (e:Employee {name: '김서준'}) REMOVE e:Manager    // 레이블 하나
```

- 속성은 **점**(`e.phone`), 레이블은 **콜론**(`e:Manager`)으로 적습니다. `SET` 과 표기가 같습니다.
- 떼어 낸 속성을 읽으면 에러가 아니라 **`None`** 이 옵니다(1-5 에서 본 것과 같습니다). 애초에 없던 속성과 구별되지 않습니다. 그래서 아래 그림처럼 "속성을 떼는 것"과 "값을 `null` 로 두는 것"이 같은 뜻이고, `SET e.phone = null` 도 결과가 같습니다.
- **노드와 관계 자체는 그대로 남습니다.** `REMOVE` 가 건드리는 것은 그 안의 속성·레이블뿐입니다.

이름이 비슷한 **`DELETE`** 는 노드나 관계 **자체**를 지웁니다(다음 장에서 배웁니다). 헷갈리기 쉬우니 경계를 미리 정리해 둡니다.

| 문장 | 사라지는 것 | 남는 것 |
|---|---|---|
| `REMOVE e.phone` | 그 속성 한 칸 | 노드, 다른 속성, 붙어 있던 관계 |
| `REMOVE e:Manager` | 레이블 하나 | 노드, 속성, 관계 |
| `DELETE e` | 노드 자체 | 반대편 노드들(다음 장에서) |

무엇을 떼어 내는지에 따라 적는 자리가 다릅니다. 점 뒤는 속성, 콜론 뒤는 레이블입니다.

<img src="images/remove-property-label.png" width="820">

In [ ]:
# 3-1 에서 붙인 연락처를 다시 뗀다. 지울 대상이 속성이므로 점을 찍어 e.phone 이라고 적는다
run_cypher("MATCH (e:Employee {name: '김서준'}) REMOVE e.phone")
# 확인: 떼어 낸 속성을 읽으면 에러가 아니라 None 이 온다. 옆의 years 는 그대로 남아 있다
print(run_cypher(
    "MATCH (e:Employee {name: '김서준'}) RETURN e.phone AS phone, e.years AS years"
))

> 레이블도 같은 방식으로 뗍니다. 뗄 레이블이 있어야 하니, 먼저 `SET` 으로 하나 붙여 봅니다. 1-1 에서 류지호에게 레이블 둘을 **만들 때부터** 붙였다면, 여기서는 이미 있는 노드에 **나중에** 붙입니다.

In [ ]:
# 김서준이 팀장을 맡았다. 레이블을 붙이는 것도 SET 이다. 속성과 달리 점이 아니라 콜론으로 적는다
run_cypher("MATCH (e:Employee {name: '김서준'}) SET e:Manager")
# 확인: 같은 노드가 이제 :Manager 로도 걸린다. Employee 레이블이 떨어진 게 아니라 하나 더 붙은 것이다
print("Manager 로 걸리는 노드:", len(run_cypher("MATCH (m:Manager) RETURN m")))

In [ ]:
# 팀장 임기가 끝났다. 레이블만 떼어 낸다. 노드도 속성도 관계도 건드리지 않는다
run_cypher("MATCH (e:Employee {name: '김서준'}) REMOVE e:Manager")
print("Manager 로 걸리는 노드:", len(run_cypher("MATCH (m:Manager) RETURN m")))
print("김서준 노드:", len(run_cypher(
    "MATCH (e:Employee {name: '김서준'}) RETURN e")))   # 레이블만 뗐으니 노드는 그대로다

### 🖐️ 함께 따라하기: 이메일을 도로 떼고, 대표 레이블 붙였다 떼기

다시 **동아리 그래프**입니다. 서지안이 이메일 공개를 원하지 않아 3-1 에서 넣은 `email` 속성을 도로 떼기로 했습니다. `REMOVE` 로 그 속성만 지운 뒤, `email` 과 `grade` 를 함께 `RETURN` 해 출력하세요.

**확인 기준**: 출력이 `[{'email': None, 'grade': 2}]` 한 줄이면 됩니다. `email` 은 칸이 사라져 `None` 으로 오고, 3-1 에서 고친 `grade` 는 그대로 남아 있습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 서지안(Student)을 MATCH 해 REMOVE 로 email 속성만 뗀다(점을 찍어 적는다)
# 2) 이어서 서지안의 email 과 grade 를 함께 RETURN 해 출력한다

이어서 레이블도 붙였다 떼어 봅니다. 서지안이 이번 학기 **동아리 대표**를 맡았다가 임기가 끝났다고 해 봅시다. 대표를 맡는 동안에는 `Leader` 레이블이 하나 더 붙고, 임기가 끝나면 그 레이블만 떨어집니다.

In [ ]:
# 🖐️ 함께 따라하기 (이어서)
# 3) 서지안에게 SET 으로 Leader 레이블을 붙인다(점이 아니라 콜론)
# 4) Leader 로 걸리는 노드 수를 len() 으로 세어 출력한다

In [ ]:
# 🖐️ 함께 따라하기 (이어서)
# 5) REMOVE 로 Leader 레이블만 뗀다
# 6) 다시 세어 0 인지, 서지안 노드는 그대로 1 인지 출력해 확인한다

### ✅ 바로 확인 퀴즈

**1.** `REMOVE e.phone` 을 실행한 뒤 `RETURN e.phone` 하면 무엇이 올까요?

<details><summary>정답 보기</summary>

**`None`** 이 옵니다(파이썬 기준). 속성 칸 자체가 사라졌기 때문에, 애초에 그 속성이 없던 노드와 똑같이 읽힙니다. 에러가 나지는 않습니다.

</details>

**2.** 노드는 남기고 레이블 하나만 떼려면 `DELETE` 와 `REMOVE` 중 무엇을 쓰나요?

<details><summary>정답 보기</summary>

**`REMOVE e:레이블`** 입니다. `REMOVE` 는 노드 안의 속성·레이블만 건드리고 노드 자체는 남깁니다. `DELETE` 는 노드나 관계 자체를 지우는 낱말입니다(다음 장).

</details>

**3.** 연락처를 없애려고 `MATCH (e:Employee {name: '김서준'}) DELETE e.phone` 이라고 썼습니다. 무엇이 잘못됐을까요?

<details><summary>정답 보기</summary>

`DELETE` 는 **노드나 관계를 담은 변수**를 받습니다. `e.phone` 은 속성 값이라 지울 대상이 될 수 없어 거부됩니다. 속성을 떼어 내는 것은 `REMOVE e.phone` 입니다.

</details>

---
# 4. 삭제: 잘못 만든 것 되돌리기

실습을 하다 보면 잘못 만든 노드나 관계가 남습니다. 지금까지는 맨 위 초기화 셀로 **전부** 지우고 처음부터 다시 만들었지만, 그건 못 하나 뽑으려고 집을 허무는 셈입니다. 하나만 골라 지우는 법을 두 걸음으로 봅니다. 4-1 은 **무엇을 지울지 고르는 법**, 4-2 는 **관계가 붙어 있을 때**입니다.

## 4-1. DELETE: 변수에 담은 그것이 지워진다

### 왜 필요할까요?
지우기의 첫 관문은 "무엇을" 입니다. 노드를 지울지 화살표만 지울지는 **MATCH 의 어느 자리에 변수를 붙였는지**로 정해집니다.

### 문법: MATCH 로 잡아 DELETE
```text
MATCH (p:Project {name: '알림봇'}) DELETE p                        // 노드 하나
MATCH (:Employee {name: '한소희'})-[r:ASSIGNED_TO]->() DELETE r    // 관계 하나
```

- 지우려면 **먼저 `MATCH` 로 찾아 변수에 담습니다.** `DELETE` 는 그 변수가 가리키는 것을 지웁니다.
- 관계도 같습니다. 관계에 변수를 붙여(`-[r:ASSIGNED_TO]->`) `DELETE r` 이라고 적으면 **화살표만 사라지고 양쪽 노드는 그대로** 남습니다.

무엇에 변수를 붙였느냐가 지울 대상을 정합니다.

<img src="images/delete-scope.png" width="820">

In [ ]:
# 1) 관계가 붙어 있지 않은 노드는 DELETE 하나로 지워진다
# 앞 절에서 만든 알림봇 프로젝트는 아직 아무 화살표도 붙지 않았다
run_cypher("MATCH (p:Project {name: '알림봇'}) DELETE p")
# 확인: 알림봇 노드가 0 이어야 한다
print("남은 알림봇 노드:", len(run_cypher("MATCH (p:Project {name: '알림봇'}) RETURN p")))

> 관계만 지우는 것도 자주 씁니다. 한소희의 앱개편 **배정을 취소**해 봅니다. 배정 관계만 사라지고 한소희와 앱개편 **노드는 둘 다 그대로** 남습니다.

In [ ]:
# 2) 관계만 지운다. 지울 대상이 관계이므로 관계에 변수 r 을 붙여 DELETE r 이라고 적는다
# 끝의 () 는 '어떤 노드든 상관없다'는 뜻이다. 여기서는 앱개편으로 가는 화살표가 하나뿐이라 이걸로 충분하다
run_cypher("MATCH (:Employee {name: '한소희'})-[r:ASSIGNED_TO]->() DELETE r")
# 확인: 배정 관계는 0 개, 한소희 노드는 그대로 1 개
print("한소희의 배정 관계:", len(run_cypher(
    "MATCH (:Employee {name: '한소희'})-[r:ASSIGNED_TO]->() RETURN r")))
print("한소희 노드:", len(run_cypher(
    "MATCH (e:Employee {name: '한소희'}) RETURN e")))

### 🖐️ 함께 따라하기: 참가를 취소하기

다시 **동아리 그래프**입니다. **서지안**이 **출사모임** 참가를 취소했습니다. 학생도 행사도 남기고 **그 참가 관계만** 지우세요. 관계에 변수 `r` 을 붙여 `MATCH` 한 뒤 `DELETE r` 로 지웁니다. 그런 다음 서지안의 남은 `JOINS` 관계 수를 `len(run_cypher(...))` 로 세어 출력해 확인하세요.

**확인 기준**: 서지안은 봄전시회 참가만 남아 **1** 이 나옵니다. 서지안 노드도 출사모임 노드도 그대로 있습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 서지안에서 출사모임으로 가는 JOINS 관계에 변수 r 을 붙여 MATCH 한다
# 2) 이어서 DELETE r 을 적는다(노드가 아니라 r 을 지운다)
# 3) 서지안의 남은 JOINS 관계를 MATCH 해 len() 으로 세어 출력한다

### ✅ 바로 확인 퀴즈

**1.** `MATCH (s:Student {name: '서지안'})-[r:JOINS]->(v:Event) DELETE s` 라고 쓰면 무엇이 지워질까요?

<details><summary>정답 보기</summary>

**서지안 노드**를 지우려 합니다. `DELETE` 뒤에 적은 변수가 `s` 이기 때문입니다. 관계만 지우려면 `DELETE r` 이라고 적어야 합니다. 참고로 서지안에게는 관계가 붙어 있어 이 문장은 거부됩니다(그 이야기는 4-2 에서 다룹니다).

</details>

**2.** 관계를 지우면 양쪽 노드는 어떻게 되나요?

<details><summary>정답 보기</summary>

**그대로 남습니다.** 지워진 것은 둘을 잇던 화살표뿐입니다. 두 노드는 이제 서로 이어져 있지 않을 뿐 그래프에 그대로 있습니다.

</details>

---
## 4-2. DETACH DELETE: 관계가 붙어 있을 때

### 왜 필요할까요?
**노드에 관계가 하나라도 붙어 있으면 그냥 `DELETE` 로는 지워지지 않습니다.** 지우면 화살표 한쪽 끝이 허공에 남기 때문입니다. 그때 쓰는 것이 **`DETACH DELETE`** 입니다. "붙어 있는 관계까지 함께 떼어 내고 지운다"는 뜻이죠.

### 문법: DETACH DELETE
```text
MATCH (e:Employee {name: '문가영'}) DETACH DELETE e
```

1-6 에서 만든 문가영에게는 기획팀으로 가는 `WORKS_IN` 화살표가 붙어 있습니다. 그냥 `DELETE` 로 지우려 하면 어떻게 되는지 직접 봅니다.

두 문장의 차이는 이렇습니다.

<img src="images/detach-delete.png" width="820">

In [ ]:
# 3) 관계가 붙은 노드를 그냥 DELETE 하면 데이터베이스가 거부한다
# ConstraintError 만 좁혀 잡는다. Exception 으로 받으면 오타·연결 끊김까지 '정상'으로 삼킨다
try:
    run_cypher("MATCH (e:Employee {name: '문가영'}) DELETE e")
    print("지워졌습니다. 붙은 관계가 없었다는 뜻입니다")   # 이쪽이 찍히면 관계가 이미 없었던 것이다
except ConstraintError as error:
    print("거부됐습니다")   # 이쪽이 찍혀야 정상이다
    # 에러 메시지가 길어 읽기 힘들다. '}' 뒤부터 앞부분만 잘라 보여 준다
    print("  ", str(error).split("}")[1].strip()[:140])   # 관계가 남아 있어 못 지운다는 안내

In [ ]:
# 4) DETACH 를 붙이면 붙어 있던 관계까지 함께 떼어 내고 노드를 지운다
run_cypher("MATCH (e:Employee {name: '문가영'}) DETACH DELETE e")
# 확인: 문가영은 사라졌지만 기획팀 노드는 남는다. 지운 것은 문가영과 그에 붙은 화살표뿐이다
print("문가영 노드:", len(run_cypher("MATCH (e:Employee {name: '문가영'}) RETURN e")))
print("기획팀 노드:", len(run_cypher("MATCH (t:Team {name: '기획팀'}) RETURN t")))

> ⚠️ **`DELETE` 앞의 `MATCH` 가 곧 지울 범위입니다.** 조건을 빠뜨리면 지울 생각이 없던 것까지 지워집니다. 맨 위 초기화 셀의 `MATCH (n) DETACH DELETE n` 이 바로 **조건이 하나도 없는** 문장이라 데이터베이스를 통째로 비우는 것이었죠. 지우기 전에 같은 `MATCH` 를 **`RETURN` 으로 먼저 돌려** 무엇이 걸리는지 눈으로 확인하는 습관을 들이세요. 지운 것은 되돌릴 수 없습니다.

### 🖐️ 함께 따라하기: 자퇴한 학생 지우기

다시 **동아리 그래프**입니다. 앞 절에서 넣은 신입생 **하지원**이 자퇴해 그래프에서 빼야 합니다. 하지원에게는 노래동아리로 가는 `BELONGS_TO` 관계가 붙어 있으니 **`DETACH DELETE`** 로 지우세요. 그런 다음 남은 `Student` 노드 수를 `len(run_cypher(...))` 로 세어 출력하세요.

**확인 기준**: 남은 학생은 **6명**입니다. 시드의 다섯 명에 1-3 에서 만든 차은우가 더해진 수이고, 방금 지운 하지원만 빠집니다(노래동아리 노드는 그대로 남습니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 하지원(Student)을 찾아 DETACH DELETE 한다
# 2) 남은 Student 노드를 MATCH 해 len() 으로 세어 출력한다

### ✅ 바로 확인 퀴즈

**1.** 관계가 붙어 있는 노드를 그냥 `DELETE` 로 지울 수 없게 막아 둔 이유는 무엇일까요?

<details><summary>정답 보기</summary>

노드만 사라지면 그 노드로 향하던 **화살표의 한쪽 끝이 허공에 남기** 때문입니다. 그런 관계는 그래프에 있을 수 없으므로, 데이터베이스가 아예 거부하고 `DETACH DELETE` 를 쓰게 합니다.

</details>

**2.** `MATCH (a)-[r:R]->(b) DELETE r` 과 `MATCH (a) DETACH DELETE a` 는 각각 무엇이 남고 무엇이 사라지나요?

<details><summary>정답 보기</summary>

앞은 **화살표만** 사라지고 노드 `a`·`b` 는 둘 다 남습니다. 뒤는 노드 `a` 와 **`a` 에 붙은 관계 전부**가 사라지고, 반대편 노드들은 남습니다.

</details>

---
## 🚀 응용 클론코딩: 새 직원을 넣고 조회까지

새 직원 **오세훈**(역할 `프론트엔드`)을 노바랩스에 합류시켰다가, 배정 하나를 되돌리는 데까지 한 번에 해 봅니다. 오늘 배운 것이 순서대로 다 나옵니다.

1. 오세훈(`Employee`, `role='프론트엔드'`) 노드를 `CREATE` 하고, **이미 있는** 디자인팀을 `MATCH` 로 찾아 `WORKS_IN` 으로 잇는다. 그런 다음 오세훈의 소속 팀을 조회해 별칭 `team` 으로 출력한다.
2. 오세훈을 **앱개편** 프로젝트에 배정한다. 배정 관계에 `{since: 2026, hours: 15}` 를 붙이고, **같은 문장에서 `RETURN`** 으로 별칭 `since`·`hours` 를 돌려받아 출력한다.
3. 배정이 잘못됐다. 그 **배정 관계만** `DELETE` 로 지우고(노드는 남긴다), 오세훈의 남은 배정 관계 수를 `len()` 으로 세어 출력한다(0 이어야 한다).

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) 오세훈(Employee, role='프론트엔드') 노드를 CREATE 한다
#    디자인팀은 이미 있으므로 두 노드를 MATCH 해 (e)-[:WORKS_IN]->(t) 관계를 CREATE 한다
#    오세훈의 소속 팀을 MATCH 해 t.name 을 별칭 team 으로 RETURN 하고 출력한다
# 2) 오세훈과 앱개편을 MATCH 해 (e)-[r:ASSIGNED_TO {since: 2026, hours: 15}]->(p) 를 CREATE 하고
#    같은 문장 끝에 RETURN 을 이어 써 r.since·r.hours 를 별칭 since·hours 로 돌려받아 출력한다
# 3) 그 배정 관계에 변수를 붙여 MATCH 한 뒤 DELETE 로 지우고, 남은 배정 관계 수를 세어 출력한다

---
## 이번 강의 정리

| 절 | 문법 | 핵심 |
|---|---|---|
| 1-1 | `CREATE (:레이블 {속성: 값})` | 동그라미 하나를 만든다 |
| 1-1 | `CREATE (p:레이블 {…}) RETURN p.속성 AS 별칭` | 변수를 붙이면 그 자리에서 돌려받는다 |
| 1-2 | `years: 5` · `name: '5'` · `active: true` · `date('2024-05-01')` | 어떻게 적느냐가 자료형을 정한다 |
| 1-2 | `날짜값.year` · `duration.between(앞, 뒤)` | 날짜로 적어야 성분·기간을 다룰 수 있다 |
| 1-3 | `(:A:B {…})` · `MATCH (x:A:B)` | 레이블을 나란히 적으면 **AND** 다 |
| 1-4 | `MATCH (a),(b) CREATE (a)-[:종류]->(b)` | 이미 있는 둘을 찾아서 잇는다 |
| 1-5 | `-[r:종류 {속성: 값}]->` … `r.속성` | 그 연결에만 해당하는 값은 화살표가 든다 |
| 1-6 | `CREATE (a:A {…})-[:R]->(b:B {…})` | 양쪽이 **모두 새것**일 때만 |
| 2-1 | `MATCH (e:레이블) RETURN e.속성 AS 별칭` | 찾아서 원하는 값을 돌려받는다 |
| 2-1 | `(e:레이블 {속성: 값})` | 패턴 안 속성 map 으로 범위를 좁힌다 |
| 2-2 | `RETURN e` | 노드가 통째로 온다(`row['e']['name']`) |
| 3-1 | `MATCH (e:레이블 {…}) SET e.속성 = 값` | 있으면 덮어쓰고 없으면 새로 붙는다 |
| 3-1 | `SET e.속성1 = 값1, e.속성2 = 값2` | 쉼표로 여러 값을 한 문장에 |
| 3-2 | `REMOVE e.속성` · `REMOVE e:레이블` | 칸만 떼어 내고 노드는 남는다 |
| 4-1 | `MATCH (n:레이블 {…}) DELETE n` | 붙은 관계가 없는 노드는 이걸로 지운다 |
| 4-1 | `MATCH ()-[r:종류]->() DELETE r` | 화살표만 사라지고 노드는 남는다 |
| 4-2 | `MATCH (n:레이블 {…}) DETACH DELETE n` | 붙은 관계까지 함께 떼어 내고 지운다 |

- Cypher 는 **그림을 그리듯** 쓰는 언어입니다. `()` 는 노드, `-[:R]->` 는 방향 관계.
- `AS 별칭` 이 결과 dict 의 **키**를 정합니다(`row['별칭']`).
- **값이 어디에 붙는지**로 노드 속성과 관계 속성을 가릅니다. 그 연결 하나에만 해당하면 관계 속성.
- `SET` 은 **값을 넣고**, `REMOVE` 는 **칸을 떼며**, `DELETE` 는 **노드·관계 자체를 지웁니다.** 고칠 때 노드를 지웠다 다시 만들면 붙어 있던 관계까지 사라집니다.
- `SET`·`REMOVE`·`DELETE` 가 미치는 범위는 모두 앞의 `MATCH` 가 정합니다. 조건을 빠뜨리면 걸린 것이 **전부** 바뀌거나 지워집니다.
- `DELETE` 는 되돌릴 수 없습니다. 지우기 전에 같은 `MATCH` 를 `RETURN` 으로 먼저 돌려 보세요.
- 쿼리가 길어지면 삼중따옴표로 **절마다 줄을 바꿔** 쓰고, 설명은 `//` 주석으로 답니다.

## ⏭️ 예고: 다음 시간

속성 map 만으로는 "근속 **3년 이상**" 같은 조건을 걸 수 없습니다. 다음 시간에는 **WHERE** 로 비교·조합 조건을 걸고, **MERGE** 로 "있으면 그대로, 없으면 만들기"(멱등)를 배웁니다. 오늘 "한쪽이 이미 있으면 한 문장 생성을 쓰면 안 된다"고 했던 고민이 MERGE 로 사라집니다. 그리고 여러 노드를 잇는 **다단 관계 패턴**도 다룹니다.

수고하셨습니다!